In [ ]:
# Configuration

# Which task to run TSP on?
task = 1  # 1, 2, 3, 4

depot_index = 86  # In order of the addresses file
vehicle_capacity = 499

In [ ]:
# Importing everything that is necessary
import json
import math

import folium
import gurobipy as gp

from data_helpers import read_customers_csv, read_distance_matrix_csv

In [ ]:
# Reading our data
dist, addresses = read_distance_matrix_csv()
customers_data = read_customers_csv()

In [ ]:
# Mapping new index to old index, to retrieve it when needed
new_to_old_index = {
    0: depot_index
}

In [ ]:
# Let's calculate in, which address indexes we have customers and what is the total
#   capacity in each of the locations
customer_indexes = []
total_capacities_in_customer_locations = []

customer_addresses = [customer["address"] for customer in customers_data]
for i, address in enumerate(addresses):
    if address in customer_addresses:
        customer_indexes.append(i)
        total_capacities_in_customer_locations.append(
            min(
                sum([customer["volume"] for customer in customers_data if address == customer["address"]]), 
                vehicle_capacity
            )
        )

In [ ]:
# Depending on the task chosen at the top, we need to limit the amount of data
if task == 1:
    lon_right_limit = 22.3
elif task == 2:
    lon_right_limit = 23.6
elif task == 3:
    lon_right_limit = 24.6
else:  # task == 4
    # Task 4 contains all locations and longitude of 30 is larger than for any of the
    #   locations
    lon_right_limit = 30.0

In [ ]:
chosen_indexes = []
chosen_customers = []
for i, idx in enumerate(customer_indexes):
    customer_address = addresses[idx]
    # Depending on the chosen task, take only certain customers
    if customer_address[1] <= lon_right_limit:
        chosen_indexes.append(i)
        chosen_customers.append(idx)
num_customers = len(chosen_indexes)

In [ ]:
# Mapping the task
lithuania_map = folium.Map(location=[55, 24], zoom_start=8)

# Show depot marker
folium.Marker(
    location=addresses[depot_index],
    icon=folium.Icon(icon="home", color="blue")
).add_to(lithuania_map)

# Show a marker for each of the customers
for i, customer in enumerate(chosen_customers):
    folium.Marker(
        location=addresses[customer],
        icon=folium.Icon(icon="user", color="orange", prefix="fa")
    ).add_to(lithuania_map)
    # Meanwhile also fill the new to old index mapping
    new_to_old_index[i + 1] = customer

lithuania_map

In [ ]:
customers = list(range(1, num_customers + 1))  # 1, 2, ..., num_customers
locations = [0] + customers  # 0, 1, 2, ..., num_customers
# Edges between all locations that aren't the same
edges = [(i, j) for i in locations for j in locations if i != j]

# Sum of distances between locations will be minimized
distances = {
    (i, j): dist[(new_to_old_index[i], new_to_old_index[j])]
    for i in locations
    for j in locations
    if i != j
}

# Let's take only capacities for the customers that were chosen
capacities = [total_capacities_in_customer_locations[chosen_index] for chosen_index in chosen_indexes]
# Let's set them as the demands
demands = {i + 1: capacity for i, capacity in enumerate(capacities)}
# The depot has no demand
demands[0] = 0

In [ ]:
# Helper functions for the main code below


def read_credentials(file_path):
    with open(file_path, "r") as file:
        return json.load(file)

In [ ]:
# This is the real entry point of the program

# Reading credentials for your WLS license
credentials = read_credentials("credentials.json")

# Parsing them to the necessary format
params = {
    "WLSACCESSID": credentials["access_id"],
    "WLSSECRET": credentials["secret"],
    "LICENSEID": credentials["license_id"],
}

# Creating your own environment gives you more control over Gurobi licensing, and it is
#   necessary when running instances of the size we are looking at
env = gp.Env(params=params)

# Naming the model and specifying our environment
m = gp.Model("CVRP", env=env)

# Let's define our decision variables
# If we do not tell GRB.BINARY specifically, it will allow values like 0.67, and we
#   can't use only 67% of a specific road
# x[i][j] = 1 means to use a road, 0 means to not use a road
x = m.addVars(edges, vtype=gp.GRB.BINARY, name="edge")

# We want to minimize linear combination of coefficients, that is distance of the tour
m.setObjective(x.prod(distances), gp.GRB.MINIMIZE)

# For each source location the sum should be 1, that is, only 1 destination location
#   should be supplied
m.addConstrs(x.sum(i, "*") == 1 for i in customers)
# For each destination location the sum should be 1, that is, only 1 source location
#   should be supplied
m.addConstrs(x.sum("*", i) == 1 for i in customers)

# Let's set the minimum amount of vehicles, this helps performance
total_demand = 0
for i in customers:
    total_demand += demands[i]
min_vehicles_needed = math.ceil(total_demand / vehicle_capacity)
# Let's set the constraint
m.addConstr(x.sum(0, "*") >= min_vehicles_needed)

# Let's add a continuous variable in range 0 to vehicle_capacity
z = m.addVars(edges, lb=0, ub=vehicle_capacity)

# You can't take any demand from depo
for i in customers:
    z[0, i].UB = 0

# Demand need to be conserved
m.addConstrs(z.sum("*", j) + demands[j] == z.sum(j, "*") for j in customers)
# Lower bound
m.addConstrs(z[i, j] >= demands[i] * x[i, j] for i in customers for j in locations if i != j)
# Upper bound
m.addConstrs(z[i, j] <= (vehicle_capacity - demands[j]) * x[i, j] for i in customers for j in locations if i != j)

# Solve the model
m.Params.TimeLimit = 30 * 60  # In seconds, that is 30 minutes
# Runs the optimization engine
m.optimize()

In [ ]:
# Output the structure of the model
m.write("3_CVRP.lp")

# Find currently selected edges
selected_edges = [(i, j) for (i, j) in x.keys() if x[i, j].X > 0.5]

# For each source location, let's find all possible destination locations, for depot
#   there will be many, but for others there will be one
next_location = {}
for i, j in selected_edges:
    if i == 0:  # If the source location is the depot
        # If the depot list isn't created yet, create it
        if 0 not in next_location.keys():
            next_location[0] = []
        next_location[0].append(j)
    else:
        next_location[i] = j

# How many cars are going out of the depot?
depot_next_location = next_location[0] 
depot_outgoing = len(depot_next_location)
print(f"{depot_outgoing} cars are required:")
# Save the route, so we can also color them later
solution_routes = {}
for car_number, first_location in enumerate(depot_next_location):
    print(f"Car {car_number}: 0 -> ", end="")
    solution_routes[car_number] = [0]
    car_capacity_taken = 0
    location_pointer = first_location
    while location_pointer != 0:
        print(f"{location_pointer} -> ", end="")
        solution_routes[car_number].append(location_pointer)
        car_capacity_taken += demands[location_pointer]
        location_pointer = next_location[location_pointer]
    print(f"0, capacity = {car_capacity_taken} of {vehicle_capacity}")
    solution_routes[car_number].append(0)

In [ ]:
# Mapping the solution to a folium map
all_colors = ["beige", "darkblue", "green", "red", "lightred", "purple", "cadetblue", "pink", "darkgreen", "lightblue", "gray", "lightgreen", "black", "blue", "lightgray", "white", "orange", "darkpurple", "darkred"]

lithuania_map = folium.Map(location=[55, 24], zoom_start=8)

# Show depot marker
folium.Marker(
    location=addresses[depot_index],
    icon=folium.Icon(icon="home", color="blue")
).add_to(lithuania_map)

# Show each route in a different color
for route_id, route in solution_routes.items():
    for location_id in range(len(route) - 1):
        i = route[location_id]
        j = route[location_id + 1]
        folium.PolyLine([addresses[new_to_old_index[i]], addresses[new_to_old_index[j]]], color=all_colors[route_id]).add_to(lithuania_map)

lithuania_map

In [ ]:
# Cleanup
m.dispose()  # Free all resources associated to this model
gp.disposeDefaultEnv()  # Disposes of default environment created by Gurobi